# Prompt Optimizer — Colab 预训练

在 T4 GPU 上加速训练你的 10M 参数 GPT 模型。
总耗时约 **10 分钟**。

## 第 1 步：挂载 Google Drive

运行下面这个单元格，会弹出一个授权链接，点进去授权，复制验证码粘贴回来。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 第 2 步：复制项目到 Colab

In [ ]:
import os, shutil

# 你的项目在 Google Drive 的路径
PROJECT_PATH = "/content/drive/MyDrive/prompt-optimizer"

if os.path.exists(PROJECT_PATH):
    if os.path.exists('/content/prompt-optimizer'):
        shutil.rmtree('/content/prompt-optimizer')
    shutil.copytree(PROJECT_PATH, '/content/prompt-optimizer')
    os.chdir('/content/prompt-optimizer')
    print(f"项目已复制到 Colab: {os.getcwd()}")
    !ls -la
else:
    print("错误：未找到项目文件夹")
    print("请先把 prompt-optimizer 文件夹上传到 Google Drive 的 MyDrive 根目录")

## 第 3 步：安装依赖

In [ ]:
!pip install torch tokenizers tqdm numpy sentencepiece datasets 2>&1 | tail -5
print("\n安装完成")

## 第 4 步：检查 GPU

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
!nvidia-smi

## 第 5 步：加载语料

如果语料已经在项目中（你在本地已经执行过数据收集），这步会自动跳过。

In [ ]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

corpus_path = "data/full_corpus.txt"
if not os.path.exists(corpus_path) or os.path.getsize(corpus_path) < 100000:
    print("正在从镜像站下载中文百科语料...")
    from datasets import load_dataset
    from tqdm import tqdm
    ds = load_dataset("wikimedia/wikipedia", "20231101.zh", split="train", streaming=True)
    with open("data/wiki_cn.txt", "w", encoding="utf-8") as f:
        count = 0
        for ex in tqdm(ds):
            text = ex.get("text", "")
            if text and len(text.strip()) > 100:
                line = text.replace("\n", " ").replace("\r", " ").strip()
                f.write(line + "\n")
                count += 1
                if count >= 5000:
                    break
    print(f"下载完成: {count} 篇")
    !cat data/*.txt > data/full_corpus.txt 2>/dev/null
    print(f"合并完成: {os.path.getsize(corpus_path)/1024:.0f} KB")
else:
    print(f"语料已存在 ({os.path.getsize(corpus_path)/1024:.0f} KB)")

## 第 6 步：准备 Tokenizer

如果本地已经训过 tokenizer，这步跳过。

In [ ]:
if not os.path.exists("tokenizer/prompt_opt_tokenizer.json"):
    !python tokenizer/train_tokenizer.py \
        --corpus-path data/full_corpus.txt \
        --output tokenizer/prompt_opt_tokenizer.json \
        --vocab-size 8000
else:
    print("Tokenizer 已存在，跳过")

## 第 7 步：预训练（核心！）

T4 GPU 上约 **10 分钟**。训练过程中每 100 步打印一次 loss。

In [ ]:
!python train/trainer.py \
    --tokenizer tokenizer/prompt_opt_tokenizer.json \
    --train-corpus data/full_corpus.txt \
    --max-steps 5000 \
    --batch-size 16 \
    --grad-accum-steps 4 \
    --lr 3e-4 \
    --warmup-steps 200 \
    --output-dir checkpoints \
    --log-dir runs \
    --log-interval 100 \
    --save-interval 1000 \
    --device cuda

## 第 8 步：保存结果到 Google Drive

训练完成后把 checkpoint 复制回你的 Google Drive，退出 Colab 后不会丢失。

In [ ]:
import os, shutil
out = "/content/drive/MyDrive/prompt-optimizer/checkpoints"
os.makedirs(out, exist_ok=True)
for f in os.listdir("checkpoints"):
    if f.endswith(".pt"):
        shutil.copy(f"checkpoints/{f}", f"{out}/{f}")
        size_mb = os.path.getsize(f"{out}/{f}") / 1024 / 1024
        print(f"  {f}  ({size_mb:.1f} MB)")
print(f"\n已保存到 Google Drive")

## 完成！

回到本地后执行：

```bash
# 先把 checkpoint 从 Google Drive 下载回本地项目
# 然后
python tokenizer/extend_tokenizer.py --input-tokenizer tokenizer/prompt_opt_tokenizer.json
python data/collect_sft.py --task-types code_writing data_analysis copywriting --num-per-task 30
python train/sft_trainer.py --tokenizer tokenizer/sft_tokenizer.json --sft-data data/sft_data.jsonl --pretrain-checkpoint checkpoints/checkpoint-latest.pt --epochs 5
```